# How Large Language Models Work

This notebook is an interview-focused deep dive into LLM mechanics, architecture families, scaling behavior, and practical deployment trade-offs.

Each section is written in long-form style so you can practice clear, layered explanations.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. LLM Objective in One Equation

A language model predicts the next token conditioned on prior tokens: \(P(x_{t+1}\mid x_{1:t})\). Repeating this step generates full responses. This objective is simple, but at scale it captures useful statistical structure for language, coding, and reasoning-style tasks.

In interviews, avoid saying the model is a lookup table. It is better described as a high-dimensional probabilistic function that maps context to token distributions.

---

## 2. Tokenization and Embedding Pipeline

Input text is tokenized into IDs, then mapped into vectors through an embedding matrix. Positional encoding is added so order information is preserved. Without positional signals, attention cannot distinguish token permutations.

This pipeline is foundational to understanding all subsequent transformer behavior. Interviewers often use it as a starting point before deeper architectural questions.

---

## 3. Transformer Block Anatomy

A decoder transformer block contains masked self-attention, an FFN, residual connections, and normalization. Attention handles cross-token dependency, and FFN adds non-linear expressivity at each position.

Residual and LayerNorm components are critical for optimization stability. Strong interview answers treat these as first-class design elements, not just implementation details.

In [ ]:
def softmax(x):
    z = x - np.max(x)
    e = np.exp(z)
    return e / e.sum()

logits = np.array([1.2, 2.6, 1.8, 1.0, 0.9, 2.2])
probs = softmax(logits)
print("Softmax probabilities:")
for i, p in enumerate(probs):
    print(f"token_{i}: {p:.4f}")
print("sum:", probs.sum())

---

## 4. Attention Mathematics and Intuition

Scaled dot-product attention is \(\text{softmax}(QK^T/\sqrt{d_k})V\). Query-key scores measure relevance and the weighted value aggregation creates context-aware representations.

In practice, attention is dynamic routing. Tokens select what to focus on at runtime, which is why transformers capture long-range dependencies effectively.

---

## 5. Multi-Head Attention

Multi-head attention splits representation space into multiple subspaces so different relation patterns can be learned in parallel. Heads are concatenated and projected, increasing expressiveness without fully independent full-size modules.

A practical interview point is trade-off framing: multi-head design increases representational diversity while controlling computation.

---

## 6. FFN, Residual, and LayerNorm Importance

FFNs contribute major model capacity and are often overlooked in beginner explanations. They transform attention outputs with non-linear mappings and contribute significantly to parameter count.

Residual pathways and normalization stabilize deep training and improve gradient flow. Mentioning these components clearly signals deeper understanding.

In [ ]:
n = 12
causal_mask = np.triu(np.ones((n, n), dtype=int), k=1)
allowed = 1 - causal_mask

plt.figure(figsize=(6, 5))
plt.imshow(allowed, cmap="Blues")
plt.title("Causal Attention Mask (1 = allowed)")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.colorbar(); plt.show()

---

## 7. Decoder-Only, Encoder-Only, Encoder-Decoder

Decoder-only architectures are optimized for autoregressive generation and dominate assistant-style text generation. Encoder-only architectures are bidirectional and strong for understanding tasks such as classification and embedding extraction.

Encoder-decoder architectures are effective for text-to-text transformation tasks like translation and summarization. Interview answers should map architecture choice to task and objective alignment.

---

## 8. Pretraining Objectives: CLM and MLM

Causal language modeling predicts next tokens from left context and aligns directly with generation. Masked language modeling predicts masked tokens from bidirectional context and is often strong for representation learning.

A strong explanation compares objective behavior, not just model brand names. Objective choice affects emergent capabilities, instruction following, and downstream task fit.

---

## 9. Training Dynamics and Optimization

Large-scale training requires distributed systems, data curation, mixed precision, and robust checkpointing. Stability techniques include learning-rate warmup, decay schedules, and gradient clipping.

Interviewers often ask what fails in real runs. Common issues are data quality drift, loss spikes, and throughput anomalies; mature answers mention monitoring and recovery protocols.

In [ ]:
vocab = ["language", "model", "attention", "token", "data", "the"]
logits = np.array([1.1, 2.5, 1.9, 0.8, 1.0, 2.2])
probs = np.exp(logits - logits.max())
probs = probs / probs.sum()
ranked = sorted(zip(vocab, probs), key=lambda x: x[1], reverse=True)
for token, p in ranked:
    print(f"{token:10s} -> {p:.4f}")
print("Greedy next token:", ranked[0][0])

---

## 10. Scaling Laws and Emergence

Scaling laws show predictable loss improvements as model size, data, and compute increase. A conceptual form is \(L(N,D)\approx AN^{-\alpha}+BD^{-\beta}+E\), showing diminishing returns but steady gains with balanced scaling.

Some capabilities appear to emerge sharply as scale increases. Interview-ready answers should treat emergence as empirical, benchmark-dependent behavior rather than guaranteed generalized intelligence.

---

## 11. Parameter Count Implications

Parameter count influences representational capacity, memory footprint, serving cost, and latency. Larger models can perform better but are more expensive to host and slower under tight latency budgets.

Good system design often uses model routing and fallback strategies rather than one fixed large model for all requests.

---

## 12. Training vs Inference

Training computes gradients and updates weights, making it compute- and memory-intensive. Inference performs forward passes only, but high request volume and long outputs still create substantial cost.

Decoding strategy changes inference behavior. Greedy decoding is deterministic, while stochastic sampling increases diversity. Beam search can improve specific tasks but adds runtime overhead.

In [ ]:
train_tokens_b = np.array([1, 5, 20, 80, 300, 1000])
params_b = np.array([0.1, 0.3, 1.3, 7, 34, 175])
synthetic_loss = 3.2 - 0.22 * np.log(train_tokens_b) - 0.18 * np.log(params_b)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_tokens_b, synthetic_loss, marker="o")
ax[0].set_xscale("log")
ax[0].set_title("Loss vs Data Scale")
ax[1].plot(params_b, synthetic_loss, marker="o", color="darkorange")
ax[1].set_xscale("log")
ax[1].set_title("Loss vs Parameter Scale")
plt.tight_layout(); plt.show()

---

## 13. Post-Training and Alignment

Base pretraining optimizes token prediction, not necessarily safe or helpful instruction following. Post-training stages such as instruction tuning and preference optimization adapt behavior toward practical assistant use.

A useful interview answer explains that alignment layers refine behavior on top of pretrained capabilities rather than replacing core language modeling ability.

---

## 14. Failure Modes and Evaluation

LLMs can fail through hallucination, brittle prompt sensitivity, and tool-use mistakes. Robust evaluation includes targeted failure probes, human rubric scoring, and ongoing production monitoring.

Benchmark scores alone are not enough for deployment confidence. Strong candidates discuss evaluation strategy as a continuous lifecycle.

---

## 15. Practical Deployment Trade-offs

Production systems must balance quality, latency, cost, and safety simultaneously. The best architecture is rarely the one with maximum raw benchmark score; it is the one that satisfies operational constraints reliably.

Interviewers appreciate candidates who can reason in trade-offs and describe fallback behavior when models fail.

## Interview Focus

### Q1. How does GPT generate text?
By autoregressive next-token prediction using decoder-only transformers with causal masking.

### Q2. What is attention?
A mechanism that computes context-dependent weighted aggregation across token representations.

### Q3. Why divide by \(\sqrt{d_k}\)?
To stabilize attention logits and keep softmax gradients in a useful range.

### Q4. BERT vs GPT difference?
BERT is encoder-only and bidirectional; GPT is decoder-only and generative.

### Q5. What is pretraining?
Large-scale self-supervised learning over broad corpora before task adaptation.

### Q6. What are scaling laws?
Empirical relationships between model/data/compute scale and loss improvement.

### Q7. Why do larger models need deployment strategy?
Higher quality may come with higher latency, memory, and cost, so routing is needed.

### Q8. Training vs inference core difference?
Training updates parameters with gradients; inference only performs forward generation.

### Q9. What are common LLM failure modes?
Hallucination, prompt brittleness, unsafe outputs, and tool-calling errors.

### Q10. How should I answer architecture questions?
Map architecture type directly to objective and downstream task requirements.

In [ ]:
models_b = np.array([1, 7, 13, 34, 70, 175])
bytes_per_param_fp16 = 2
memory_gb = models_b * 1e9 * bytes_per_param_fp16 / (1024**3)
for m, mem in zip(models_b, memory_gb):
    print(f"{int(m):>3}B params -> ~{mem:6.2f} GB fp16 weights")

---

## 16. Summary and Key Takeaways for Interviews

LLMs are next-token probabilistic systems built on transformer blocks with attention, FFN, residual, and normalization components. Their performance depends on architecture, objectives, scaling, and post-training alignment.

To interview well, explain mechanisms clearly, compare model families by task fit, and connect internal model behavior to deployment constraints such as latency, cost, and safety.